In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, WeightedRandomSampler
import numpy as np
import os

torch.set_num_threads(8)

BASE_DIR  = r"C:\Users\Rushikesh\OneDrive\CODES\SelfHealingNN"
os.chdir(BASE_DIR)

IMAGE_SIZE = 128
BATCH_SIZE = 16

# ✅ REMOVED Normalize — must stay in [0, 1] range to match VAE output
train_transforms = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.RandomAffine(degrees=0, translate=(0.05, 0.05)),
    transforms.ToTensor(),
])

test_transforms = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
])

train_dataset = datasets.ImageFolder(
    root=os.path.join(BASE_DIR, "medical_data", "train"),
    transform=train_transforms
)
test_dataset = datasets.ImageFolder(
    root=os.path.join(BASE_DIR, "medical_data", "test"),
    transform=test_transforms
)

# ⚖️ Balanced sampling for imbalanced classes
targets = np.array(train_dataset.targets)
class_counts = np.bincount(targets)
sample_weights = (1.0 / class_counts)[targets]
sampler = WeightedRandomSampler(sample_weights, len(sample_weights))

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=sampler,  num_workers=0)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f"✅ Classes : {train_dataset.classes}")
print(f"   Train   : {len(train_dataset)} | Test: {len(test_dataset)}")
print(f"⚖️  Distribution: {dict(zip(train_dataset.classes, class_counts))}")

✅ Classes : ['NORMAL', 'PNEUMONIA']
   Train   : 5216 | Test: 624
⚖️  Distribution: {'NORMAL': np.int64(1341), 'PNEUMONIA': np.int64(3875)}


In [2]:
class MedicalExpertCNN(nn.Module):
    def __init__(self):
        super(MedicalExpertCNN, self).__init__()

        # ✅ Added BatchNorm2d + extra conv block for deeper feature extraction
        self.features = nn.Sequential(
            nn.Conv2d(1,  32,  3, padding=1), nn.BatchNorm2d(32),  nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64,  3, padding=1), nn.BatchNorm2d(64),  nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(128,256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(), nn.MaxPool2d(2),
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256 * 8 * 8, 512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, 2)
        )

    def forward(self, x):
        return self.classifier(self.features(x))

total = sum(p.numel() for p in MedicalExpertCNN().parameters())
print(f"✅ CNN ready (with BatchNorm + 4 blocks) | Parameters: {total:,}")

✅ CNN ready (with BatchNorm + 4 blocks) | Parameters: 8,778,946


In [6]:
os.makedirs(os.path.join(BASE_DIR, "models"), exist_ok=True)

medical_expert = MedicalExpertCNN()
optimizer = optim.Adam(medical_expert.parameters(), lr=5e-4, weight_decay=1e-4)

# ✅ Stronger class weights — NORMAL is ~3x rarer, so boost it aggressively
n_normal = class_counts[0]
n_pneumonia = class_counts[1]
weight_normal = n_pneumonia / n_normal      # ~2.9
weight_pneumonia = 1.0
weights = torch.tensor([weight_normal, weight_pneumonia], dtype=torch.float32)
criterion = nn.CrossEntropyLoss(weight=weights)
print(f"⚖️  Loss weights: NORMAL={weight_normal:.2f}, PNEUMONIA={weight_pneumonia:.2f}")

scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=3, factor=0.5)

EPOCHS = 25

print(f"🚀 Training CNN for {EPOCHS} epochs...")

for epoch in range(EPOCHS):
    medical_expert.train()
    correct, total_imgs, running_loss = 0, 0, 0

    for batch_idx, (images, labels) in enumerate(train_loader):
        optimizer.zero_grad()
        outputs = medical_expert(images)
        loss    = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        predicted     = outputs.argmax(dim=1)
        total_imgs   += labels.size(0)
        correct      += (predicted == labels).sum().item()
        running_loss += loss.item()

        if batch_idx % 50 == 0:
            acc = 100 * correct / total_imgs
            print(f"  Epoch {epoch+1}/{EPOCHS} | Batch {batch_idx:>3}/{len(train_loader)} | Loss: {running_loss/(batch_idx+1):.3f} | Acc: {acc:.1f}%")

    avg_loss = running_loss / len(train_loader)
    acc = 100 * correct / total_imgs
    scheduler.step(avg_loss)
    lr = optimizer.param_groups[0]['lr']
    print(f"✅ Epoch {epoch+1}/{EPOCHS} done | Acc: {acc:.1f}% | LR: {lr:.1e}\n")

torch.save(medical_expert.state_dict(), os.path.join(BASE_DIR, "models", "medical_expert.pth"))
print("💾 Saved → models/medical_expert.pth")

⚖️  Loss weights: NORMAL=2.89, PNEUMONIA=1.00
🚀 Training CNN for 25 epochs...
  Epoch 1/25 | Batch   0/326 | Loss: 0.864 | Acc: 75.0%
  Epoch 1/25 | Batch  50/326 | Loss: 1.132 | Acc: 77.1%
  Epoch 1/25 | Batch 100/326 | Loss: 0.748 | Acc: 81.7%
  Epoch 1/25 | Batch 150/326 | Loss: 0.570 | Acc: 83.9%
  Epoch 1/25 | Batch 200/326 | Loss: 0.477 | Acc: 85.5%
  Epoch 1/25 | Batch 250/326 | Loss: 0.412 | Acc: 87.1%
  Epoch 1/25 | Batch 300/326 | Loss: 0.377 | Acc: 87.8%
✅ Epoch 1/25 done | Acc: 88.2% | LR: 5.0e-04

  Epoch 2/25 | Batch   0/326 | Loss: 0.116 | Acc: 100.0%
  Epoch 2/25 | Batch  50/326 | Loss: 0.126 | Acc: 94.0%
  Epoch 2/25 | Batch 100/326 | Loss: 0.132 | Acc: 93.9%
  Epoch 2/25 | Batch 150/326 | Loss: 0.140 | Acc: 93.4%
  Epoch 2/25 | Batch 200/326 | Loss: 0.136 | Acc: 93.6%
  Epoch 2/25 | Batch 250/326 | Loss: 0.139 | Acc: 93.4%
  Epoch 2/25 | Batch 300/326 | Loss: 0.136 | Acc: 93.5%
✅ Epoch 2/25 done | Acc: 93.7% | LR: 5.0e-04

  Epoch 3/25 | Batch   0/326 | Loss: 0.058 | 

In [7]:
medical_expert.eval()
correct, total_imgs = 0, 0
all_preds, all_labels = [], []

with torch.no_grad():
    for images, labels in test_loader:
        outputs   = medical_expert(images)
        predicted = outputs.argmax(dim=1)
        total_imgs   += labels.size(0)
        correct      += (predicted == labels).sum().item()
        all_preds.extend(predicted.tolist())
        all_labels.extend(labels.tolist())

accuracy = 100 * correct / total_imgs
print(f"🎯 Test Accuracy: {accuracy:.2f}%")

classes = train_dataset.classes
for cls_idx, cls_name in enumerate(classes):
    cls_correct = sum(p == l == cls_idx for p, l in zip(all_preds, all_labels))
    cls_total   = sum(l == cls_idx for l in all_labels)
    print(f"   {cls_name}: {cls_correct}/{cls_total} = {100*cls_correct/cls_total:.1f}%")

🎯 Test Accuracy: 87.18%
   NORMAL: 182/234 = 77.8%
   PNEUMONIA: 362/390 = 92.8%
